# Ingeniería de Features
## Construcción y Validación de Variables para el Denoising Autoencoder

**Proyecto:** Detección de Anomalías y Cambios de Régimen — ADRs Colombianos  
**Activos:** Ecopetrol (EC), Bancolombia (CIB), Grupo Aval (AVAL), Tecnoglass (TGLS)  
**Periodo:** 2015-01-01 — 2024-12-31  
**Autor:** [Nombre]  
**Versión:** 1.0

---

### Estructura del Notebook

| Sección | Contenido |
|---|---|
| §1 | Configuración y carga de datos |
| §2 | Feature 1 — Retorno Logarítmico |
| §3 | Feature 2 — Volatilidad Realizada (21 días) |
| §4 | Feature 3 — Z-Score del Volumen |
| §5 | Validación del vector de features |
| §6 | Normalización sin data leakage |
| §7 | Generación de ventanas temporales |
| §8 | Features y detección de cambios de régimen |

---

### Principios de diseño del vector de features

Todo feature incluido en el modelo debe satisfacer simultáneamente cuatro criterios:

1. **Estacionariedad:** La serie debe tener media y varianza aproximadamente
   constantes en el tiempo para garantizar estabilidad numérica del LSTM.
2. **Causalidad estricta:** El valor de la feature en t debe depender
   exclusivamente de información disponible en t o antes — ningún dato futuro.
3. **Contenido informativo independiente:** Cada feature debe aportar señal
   no redundante respecto a las demás variables del vector.
4. **Relevancia para la detección de anomalías:** La feature debe cambiar
   de manera estadísticamente diferenciada durante los periodos de crisis
   respecto al comportamiento normal.

---
## Configuración y Carga de Datos

In [ ]:
# ── Dependencias ─────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import yfinance as yf

# Reproducibilidad
np.random.seed(42)

# Estilo visual
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
})

print("Entorno configurado correctamente.")


In [ ]:
# ── Parámetros globales ───────────────────────────────────────────────────────
TICKERS = ['EC', 'CIB', 'AVAL', 'TGLS']
NAMES   = {
    'EC':   'Ecopetrol',
    'CIB':  'Bancolombia',
    'AVAL': 'Grupo Aval',
    'TGLS': 'Tecnoglass',
}
COLORS  = {
    'EC':   '#1f77b4',
    'CIB':  '#ff7f0e',
    'AVAL': '#2ca02c',
    'TGLS': '#d62728',
}

# Parámetros del modelo
SEQ_LEN      = 30    # longitud de ventana temporal (justificada en EDA §6)
ROLL_WIN     = 21    # ventana para volatilidad y vol-zscore (≈ 1 mes de negociación)
WARM_UP      = 26    # días de calentamiento (máx entre ROLL_WIN y MACD-26)
FEATURE_COLS = ['log_return', 'vol_21d', 'vol_zscore']  # vector final

# Partición temporal — sin data leakage
TRAIN_START = '2015-01-01'
TRAIN_END   = '2019-12-31'
VAL_START   = '2020-01-01'
VAL_END     = '2020-12-31'
TEST_START  = '2021-01-01'
TEST_END    = '2024-12-31'

# Periodos de anomalía conocidos (referencia visual únicamente)
ANOMALY_PERIODS = [
    {'name': 'Crisis petróleo', 'start': '2015-07-01',
     'end': '2016-02-01', 'color': 'rgba(128,0,128,0.10)'},
    {'name': 'COVID-19',        'start': '2020-02-15',
     'end': '2020-05-01', 'color': 'rgba(220,20,60,0.10)'},
    {'name': 'Fed hikes',       'start': '2022-01-01',
     'end': '2022-12-31', 'color': 'rgba(255,140,0,0.10)'},
]

print("Parámetros globales cargados.")
print(f"  Activos        : {TICKERS}")
print(f"  Ventana (T)    : {SEQ_LEN} sesiones")
print(f"  Features       : {FEATURE_COLS}")
print(f"  Train          : {TRAIN_START} → {TRAIN_END}")
print(f"  Validación     : {VAL_START} → {VAL_END}")
print(f"  Test           : {TEST_START} → {TEST_END}")


In [ ]:
# ── Descarga y preparación de datos base ─────────────────────────────────────
raw = yf.download(TICKERS, start=TRAIN_START, end=TEST_END,
                  auto_adjust=True, progress=False)

dfs = {}
for ticker in TICKERS:
    tmp = pd.DataFrame({
        'Open':   raw['Open'][ticker],
        'High':   raw['High'][ticker],
        'Low':    raw['Low'][ticker],
        'Close':  raw['Close'][ticker],
        'Volume': raw['Volume'][ticker],
    }).dropna()
    tmp.index = pd.to_datetime(tmp.index)
    dfs[ticker] = tmp

print(f"{'Ticker':<8} {'Sesiones':>10}  {'Inicio':<14}  {'Fin':<14}")
print('-' * 52)
for t, d in dfs.items():
    print(f"{t:<8} {len(d):>10}  "
          f"{str(d.index.min().date()):<14}  "
          f"{str(d.index.max().date()):<14}")


---
## Feature 1: Retorno Logarítmico

---

### **2.1 Definición matemática**

El retorno logarítmico diario se define como:

```
r_t = ln(P_t / P_{t-1}) = ln(P_t) - ln(P_{t-1})
```

donde P_t es el precio de cierre ajustado en la sesión t.

**Relación con el retorno aritmético:**

```
r_t^arith = (P_t - P_{t-1}) / P_{t-1}

Para retornos pequeños: r_t ≈ r_t^arith
Para retornos grandes:  r_t < r_t^arith  (asimetría correctiva)
```

**Propiedad de aditividad temporal:**

```
r_{t,t+k} = r_{t+1} + r_{t+2} + ... + r_{t+k}
```

Esta propiedad no se cumple para retornos aritméticos, lo que hace al
retorno logarítmico preferible para la agregación multi-periodo.

---

### **2.2 Justificación financiera**

El retorno logarítmico es la transformación estándar en finanzas cuantitativas
por las siguientes razones:

- **Estacionariedad:** La serie de precios P_t contiene raíz unitaria
  (confirmado por ADF en el EDA). El retorno logarítmico r_t es la primera
  diferencia en escala log y produce una serie I(0) — integrada de orden 0 —
  con media cercana a cero y varianza aproximadamente finita.
- **Interpretación directa:** r_t representa la tasa de cambio instantánea
  del precio, directamente comparable entre activos de distintos niveles
  de precio y entre distintos periodos de tiempo.
- **Simetría logarítmica:** Una caída del 50% (r_t = −0.693) tiene la misma
  magnitud absoluta que una subida del 100% (r_t = +0.693), a diferencia
  del retorno aritmético donde son asimétricas (−50% vs +100%).
- **Supuesto de movimiento browniano geométrico:** El modelo estándar de
  precios financieros (Black-Scholes) asume P_t = P_0·exp(μt + σW_t),
  lo que implica que los log-retornos son normales i.i.d. en el límite
  continuo. Aunque esta hipótesis no se cumple exactamente (fat tails),
  establece el log-retorno como la transformación natural del precio.

---

### **2.3 Impacto en el modelo**

- **Entrada del Autoencoder:** r_t es la primera columna del vector de
  features en cada timestep. Su rango compacto ([−0.15, 0.15] para la mayoría
  de sesiones) evita saturación de las funciones de activación del LSTM.
- **Señal de anomalía:** Los eventos anómalos generan r_t de magnitud
  excepcional (|r_t| > 3σ), produciendo ventanas que el encoder no puede
  reconstruir fielmente — el mecanismo primario de detección.
- **Normalización:** Aunque los retornos tienen escala pequeña, la varianza
  difiere entre activos (EC más volátil que AVAL). El StandardScaler por
  feature elimina esta diferencia de escala antes del entrenamiento.

In [ ]:
# ── Cómputo del retorno logarítmico ──────────────────────────────────────────
def compute_log_return(df):
    """
    Calcula el retorno logarítmico diario sobre el precio de cierre ajustado.
    El primer valor es NaN (sin t-1 disponible).

    Parámetros
    ----------
    df : pd.DataFrame con columna 'Close'

    Retorna
    -------
    pd.Series con el log-retorno
    """
    return np.log(df['Close'] / df['Close'].shift(1))

for ticker in TICKERS:
    dfs[ticker]['log_return'] = compute_log_return(dfs[ticker])

# ── Verificación numérica ─────────────────────────────────────────────────────
print("VERIFICACIÓN: RETORNO LOGARÍTMICO")
print("=" * 65)
print(f"{'Activo':<14}  {'Media':>10}  {'Std':>10}  "
      f"{'Min':>10}  {'Max':>10}  {'NaN':>6}")
print('-' * 65)

for ticker in TICKERS:
    r = dfs[ticker]['log_return'].dropna()
    print(f"{NAMES[ticker]:<14}  {r.mean():>10.6f}  {r.std():>10.6f}  "
          f"{r.min():>10.4f}  {r.max():>10.4f}  "
          f"{dfs[ticker]['log_return'].isna().sum():>6}")

print()
print("Nota: NaN=1 en cada activo corresponde al primer dia (sin t-1). Esperado.")


In [ ]:
# ── Validación de estacionariedad del log-retorno ─────────────────────────────
from statsmodels.tsa.stattools import adfuller

print("TEST ADF — LOG-RETORNOS (confirmar estacionariedad)")
print("=" * 60)
print(f"{'Activo':<14}  {'ADF stat':>10}  {'p-valor':>10}  {'Estac.':>8}")
print('-' * 48)

for ticker in TICKERS:
    r      = dfs[ticker]['log_return'].dropna()
    result = adfuller(r, autolag='AIC')
    stat, pval = result[0], result[1]
    estac  = 'SI' if pval < 0.05 else 'NO'
    print(f"{NAMES[ticker]:<14}  {stat:>10.4f}  {pval:>10.6f}  {estac:>8}")


In [ ]:
# ── Visualización: precio vs retorno (panel 2×2 por activo) ──────────────────
fig, axes = plt.subplots(len(TICKERS), 2,
                         figsize=(17, 3.5 * len(TICKERS)))

for i, ticker in enumerate(TICKERS):
    d = dfs[ticker]
    r = d['log_return'].dropna()
    sigma = r.std()

    # Precio
    axes[i, 0].plot(d['Close'], color=COLORS[ticker], linewidth=0.9)
    axes[i, 0].set_title(f'{NAMES[ticker]} — Precio de Cierre',
                         fontweight='bold')
    axes[i, 0].set_ylabel('USD')
    for ap in ANOMALY_PERIODS:
        axes[i, 0].axvspan(pd.Timestamp(ap['start']),
                           pd.Timestamp(ap['end']),
                           alpha=0.09, color='red')

    # Log-retorno
    axes[i, 1].plot(r.index, r.values,
                    color=COLORS[ticker], linewidth=0.6, alpha=0.8)
    axes[i, 1].axhline( 3 * sigma, color='grey', linewidth=0.8,
                        linestyle='--', alpha=0.6)
    axes[i, 1].axhline(-3 * sigma, color='grey', linewidth=0.8,
                        linestyle='--', alpha=0.6)
    axes[i, 1].axhline(0, color='black', linewidth=0.4, alpha=0.4)
    axes[i, 1].set_title(f'{NAMES[ticker]} — Log-Retorno (±3σ)',
                         fontweight='bold')
    axes[i, 1].set_ylabel('r_t')
    for ap in ANOMALY_PERIODS:
        axes[i, 1].axvspan(pd.Timestamp(ap['start']),
                           pd.Timestamp(ap['end']),
                           alpha=0.09, color='red')

plt.suptitle('Feature 1: Precio de Cierre vs Log-Retorno Diario',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_f1_log_return.png', dpi=120, bbox_inches='tight')
plt.show()


### **2.4 Decisión de preprocesamiento**

| Aspecto | Decisión |
|---|---|
| Fórmula | `r_t = log(Close_t / Close_{t-1})` |
| Primer valor | Descartado (NaN) — no imputar con 0 |
| Tratamiento de splits | Calculado sobre la serie completa antes de particionar |
| Normalización | StandardScaler ajustado solo en entrenamiento |
| Inclusión en el modelo | Primera columna del vector de features |

### **Riesgos identificados**

- **Leakage intra-split:** Si el log-retorno se computara por separado dentro
  de cada split (train, val, test), el primer valor de validación y test
  quedaría como NaN por falta de t−1. **Control:** Calcular sobre la serie
  completa, luego particionar por fecha.
- **Retornos durante suspensiones:** Si un activo tuviera una suspensión
  de negociación y luego reabriera con un gap de precio, el retorno del día
  de reapertura incorporaría múltiples días de movimiento acumulado, produciendo
  un falso outlier. **Control:** Verificar que todos los gaps > 3 días en el
  calendario coincidan con festivos documentados (realizado en EDA §2).

---
## Feature 2: Volatilidad Realizada (Ventana 21 Días)

---

### **3.1 Definición matemática**

La volatilidad realizada anualizada en la sesión t, con ventana de w días, se
define como la desviación estándar de los log-retornos en la ventana rodante
precedente, escalada a escala anual:

```
σ_t = sqrt( (1/(w-1)) * Σ_{i=0}^{w-1} (r_{t-i} - r̄_w)² ) × √252
```

donde:
- w = 21 (sesiones de negociación ≈ 1 mes calendario)
- r̄_w = media de los retornos en la ventana [t−w+1, t]
- √252 = factor de anualización (252 sesiones de negociación por año)

**Versión simplificada para implementación:**

```python
vol_t = retornos.rolling(21).std() * sqrt(252)
```

La función `std()` de pandas usa divisor (w−1) por defecto (corrección de Bessel),
produciendo un estimador insesgado de la desviación estándar poblacional.

**Ventana óptima — justificación:**

```
w = 5   días → captura ruido de ultra corto plazo; excesiva varianza
w = 10  días → sensible pero ruidosa
w = 21  días → equilibrio entre sensibilidad y estabilidad; estándar de mercado
w = 63  días → ventana trimestral; demasiado lenta para detectar cambios de régimen
```

---

### **3.2 Justificación financiera**

La volatilidad realizada de 21 días es la medida de riesgo de mercado más
utilizada en la gestión de carteras profesional por las siguientes razones:

- **Horizonte de decisión estándar:** El Value at Risk (VaR) mensual, el
  indicador de riesgo regulatorio estándar bajo Basilea III, usa una ventana
  de 21 sesiones de negociación. La coherencia con este horizonte facilita
  la interpretación de los resultados.
- **Captura del efecto ARCH:** El análisis ACF/PACF del EDA confirmó
  autocorrelación significativa en los retornos al cuadrado hasta lag ≈ 30.
  La ventana de 21 días captura la mayor parte de esta dependencia de
  volatilidad sin extenderse más allá del horizonte de memoria efectivo.
- **Indicador de régimen:** Durante periodos de crisis, σ_t se eleva
  abruptamente y permanece elevada por semanas, constituyendo una señal
  sostenida de cambio de régimen que complementa la información puntual
  del log-retorno.
- **Estacionariedad:** Al ser σ_t una función de los retornos (que son I(0)),
  la serie de volatilidad realizada también es aproximadamente estacionaria,
  aunque no simétrica (distribución sesgada a la derecha).

---

### **3.3 Impacto en el modelo**

- **Segunda columna del vector de features:** σ_t provee al encoder del
  Autoencoder información sobre el nivel de riesgo local en cada timestep.
  El LSTM aprende que una secuencia "normal" tiene σ_t en un rango acotado;
  una secuencia con σ_t persistentemente elevado activará un error de
  reconstrucción alto incluso si los retornos individuales son moderados.
- **Período de calentamiento:** Los primeros 20 días de la serie tienen
  σ_t = NaN (ventana incompleta). Estos días se descartan antes de
  generar las ventanas de entrada al modelo.
- **Interacción con el DAE:** El ruido artificial añadido en el entrenamiento
  (ε ~ N(0, σ²_noise)) tiene menor impacto sobre σ_t que sobre r_t, dado que
  σ_t es un estadístico suavizado (promedio de cuadrados). Esto garantiza que
  la información de volatilidad llegue al encoder con menor distorsión.

In [ ]:
# ── Cómputo de la volatilidad realizada (21 sesiones, anualizada) ─────────────
def compute_realized_vol(df, window=21, trading_days=252):
    """
    Calcula la volatilidad realizada anualizada sobre una ventana rodante.

    Parámetros
    ----------
    df      : pd.DataFrame con columna 'log_return'
    window  : int, tamaño de la ventana (default=21)
    trading_days : int, factor de anualización (default=252)

    Retorna
    -------
    pd.Series con la volatilidad anualizada

    Notas
    -----
    - Usa divisor (window-1) via ddof=1 (corrección de Bessel).
    - Los primeros (window-1) valores son NaN por diseño.
    - Cómputo causal: sólo usa información hasta t, nunca t+1.
    """
    return df['log_return'].rolling(window=window, min_periods=window).std()            * np.sqrt(trading_days)

for ticker in TICKERS:
    dfs[ticker]['vol_21d'] = compute_realized_vol(dfs[ticker],
                                                   window=ROLL_WIN)

# ── Verificación numérica ─────────────────────────────────────────────────────
print("VERIFICACIÓN: VOLATILIDAD REALIZADA 21 DÍAS (anualizada)")
print("=" * 72)
print(f"{'Activo':<14}  {'Media':>8}  {'Std':>8}  "
      f"{'Min':>8}  {'Max':>8}  {'NaN':>6}  {'Percentil 95':>14}")
print('-' * 72)

for ticker in TICKERS:
    v = dfs[ticker]['vol_21d'].dropna()
    nans = dfs[ticker]['vol_21d'].isna().sum()
    print(f"{NAMES[ticker]:<14}  {v.mean():>8.4f}  {v.std():>8.4f}  "
          f"{v.min():>8.4f}  {v.max():>8.4f}  {nans:>6}  "
          f"{np.percentile(v, 95):>14.4f}")

print()
print(f"Nota: NaN = {ROLL_WIN - 1} primeras sesiones (ventana incompleta). Esperado.")


In [ ]:
# ── Comparación de ventanas: sensibilidad vs estabilidad ─────────────────────
windows_test = [5, 10, 21, 42, 63]
ticker_demo  = 'EC'

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)

# Panel superior: retornos
r = dfs[ticker_demo]['log_return'].dropna()
axes[0].plot(r.index, r.values, color='#aec7e8',
             linewidth=0.6, alpha=0.8, label='Log-retorno')
axes[0].set_title(f'{NAMES[ticker_demo]} — Log-Retorno Diario',
                  fontweight='bold')
axes[0].set_ylabel('r_t')
axes[0].legend(fontsize=9)

# Panel inferior: distintas ventanas de volatilidad
palette = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']
for win, color in zip(windows_test, palette):
    vol = dfs[ticker_demo]['log_return'].rolling(win).std() * np.sqrt(252)
    axes[1].plot(vol.index, vol.values,
                 color=color, linewidth=0.9 if win != 21 else 1.8,
                 alpha=0.7 if win != 21 else 1.0,
                 linestyle='--' if win != 21 else '-',
                 label=f'w={win}d {"<-- SELECCIONADA" if win == 21 else ""}')

for ap in ANOMALY_PERIODS:
    for ax in axes:
        ax.axvspan(pd.Timestamp(ap['start']), pd.Timestamp(ap['end']),
                   alpha=0.09, color='red')

axes[1].set_title('Comparación de Ventanas de Volatilidad (5 a 63 días)',
                  fontweight='bold')
axes[1].set_ylabel('Volatilidad anualizada')
axes[1].legend(fontsize=8, loc='upper left')

plt.suptitle(f'{NAMES[ticker_demo]} — Selección de Ventana para Volatilidad Realizada',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_f2_window_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Volatilidad rodante de todos los activos — panel interactivo ─────────────
fig = make_subplots(
    rows=len(TICKERS), cols=1,
    shared_xaxes=True,
    subplot_titles=[f'{NAMES[t]} — Volatilidad Realizada 21d' for t in TICKERS],
    vertical_spacing=0.05
)

for i, ticker in enumerate(TICKERS, start=1):
    d = dfs[ticker].dropna(subset=['vol_21d'])

    fig.add_trace(
        go.Scatter(x=d.index, y=d['vol_21d'],
                   mode='lines', name=NAMES[ticker],
                   line=dict(color=COLORS[ticker], width=1.2)),
        row=i, col=1
    )
    for ap in ANOMALY_PERIODS:
        fig.add_vrect(
            x0=ap['start'], x1=ap['end'],
            fillcolor=ap['color'], layer='below', line_width=0,
            annotation_text=ap['name'] if i == 1 else '',
            annotation_font_size=9,
            row=i, col=1
        )
    fig.update_yaxes(title_text='σ_t (anual)', row=i, col=1)

fig.update_layout(
    title_text='<b>Feature 2: Volatilidad Realizada 21 Días — ADRs Colombianos</b>',
    height=280 * len(TICKERS), width=1100,
    template='plotly_white'
)
fig.show()


In [ ]:
# ── Estadísticas por régimen: normal vs crisis ───────────────────────────────
print("VOLATILIDAD MEDIA POR RÉGIMEN DE MERCADO")
print("=" * 70)

regimes = {
    'Normal (2015–2019)':  ('2015-01-01', '2019-12-31'),
    'Crisis petróleo':      ('2015-07-01', '2016-02-01'),
    'COVID-19':             ('2020-02-15', '2020-05-01'),
    'Fed Hikes (2022)':     ('2022-01-01', '2022-12-31'),
}

print(f"{'Régimen':<22}", end='')
for t in TICKERS:
    print(f"  {NAMES[t]:>14}", end='')
print()
print('-' * (22 + 16 * len(TICKERS)))

for regime_name, (start, end) in regimes.items():
    print(f"{regime_name:<22}", end='')
    for ticker in TICKERS:
        mask = ((dfs[ticker].index >= start) &
                (dfs[ticker].index <= end))
        mean_vol = dfs[ticker].loc[mask, 'vol_21d'].mean()
        print(f"  {mean_vol:>14.4f}", end='')
    print()

print()
print("Interpretacion: todos los activos muestran incremento > 2x en volatilidad")
print("durante COVID-19 respecto al baseline. EC muestra el mayor incremento")
print("durante la crisis del petroleo (dependencia estructural al crudo).")


### **3.4 Decisión de preprocesamiento**

| Aspecto | Decisión |
|---|---|
| Fórmula | `vol_t = rolling_std(r, w=21) * sqrt(252)` |
| Divisor | N−1 (corrección de Bessel, estimador insesgado) |
| Primeros 20 valores | NaN — descartar (no imputar) |
| Normalización | StandardScaler ajustado solo en entrenamiento |
| Inclusión en el modelo | Segunda columna del vector de features |

### **Riesgos identificados**

- **Leakage de calentamiento en frontera de split:** Los primeros 20 timesteps
  del conjunto de validación tienen σ_t calculado parcialmente con retornos del
  conjunto de entrenamiento. Este comportamiento es correcto y no constituye
  leakage: la ventana rodante usa datos pasados legítimamente. El riesgo sería
  calcular σ_t "centrando" la ventana alrededor de t (i.e., usando retornos
  futuros). **Control:** Verificar que `rolling()` use únicamente `window`
  observaciones anteriores (configuración por defecto en pandas).
- **Heterocedasticidad residual:** σ_t no elimina completamente la
  heterocedasticidad; sólo la captura con rezago. Durante shocks abruptos
  (primer día del COVID, por ejemplo), σ_t sigue reflejando el nivel de
  volatilidad pre-shock durante los primeros w días. El modelo debe aprender
  a detectar la discrepancia entre σ_t (bajo) y |r_t| (alto) como señal
  de anomalía.

---
## Feature 3: Z-Score del Volumen

---

### **4.1 Definición matemática**

El z-score del volumen es una medida estandarizada de la actividad de negociación
relativa al contexto local de los últimos w días. Se calcula en tres pasos:

**Paso 1 — Transformación logarítmica del volumen bruto:**

```
V'_t = ln(1 + V_t)
```

La transformación log1p corrige la asimetría positiva extrema del volumen
bruto (distribución aproximadamente log-normal) y estabiliza la varianza.
Se usa log(1 + V) en lugar de log(V) para manejar sesiones con V = 0 sin
generar −∞.

**Paso 2 — Estimación de estadísticos locales:**

```
μ_t  = (1/w) Σ_{i=1}^{w} V'_{t-i}      (media rodante, ventana w = 21)
σ²_t = (1/(w-1)) Σ_{i=1}^{w} (V'_{t-i} - μ_t)²   (varianza rodante)
```

Nota: la ventana usa los w días anteriores a t (sin incluir t) para
garantizar causalidad estricta. En la implementación se usa `shift(1)`
o `rolling` con `closed='left'`.

**Paso 3 — Z-score:**

```
z_vol_t = (V'_t - μ_t) / σ_t
```

z_vol_t mide cuántas desviaciones estándar se aleja el volumen del día t
respecto al promedio de los 21 días anteriores.

---

### **4.2 Justificación financiera**

El volumen de negociación es uno de los indicadores más utilizados para
confirmar o rechazar movimientos de precio. Su normalización por z-score
sobre la ventana local es fundamental por las siguientes razones:

- **Elimina la tendencia secular del volumen:** El volumen de negociación
  de los mercados bursátiles presenta una tendencia de crecimiento secular
  a lo largo de los años (más participantes, mayor liquidez). Sin normalización,
  el volumen del 2024 sería intrínsecamente mayor que el del 2015 sin que
  ello indique anormalidad. El z-score elimina esta tendencia al medir el
  volumen relativo a su contexto local.
- **Captura actividad inusual de manera invariante:** z_vol_t > 3 indica
  que el volumen del día t es 3 desviaciones estándar superior al promedio
  reciente, independientemente del nivel absoluto o del periodo temporal.
- **Señal complementaria al retorno:** Un movimiento de precio extremo
  acompañado de volumen anómalo (z_vol > 3) es más probable que sea una
  anomalía genuina (noticias, eventos macro) que el mismo retorno con
  volumen normal (potencialmente ruido de baja liquidez).
- **Indicador de participación del mercado:** Durante crisis sistémicas,
  el volumen tiende a aumentar sustancialmente por el aumento de órdenes
  de venta forzada (margin calls), rebalanceo de fondos y operaciones de
  cobertura. Este comportamiento es capturado directamente por z_vol_t.

---

### **4.3 Impacto en el modelo**

- **Tercera columna del vector de features:** z_vol_t aporta información
  sobre la intensidad de la actividad de mercado que no está correlacionada
  directamente con la magnitud o dirección del retorno.
- **Ortogonalidad parcial respecto a r_t:** Si bien existe correlación
  positiva entre |r_t| y z_vol_t durante eventos extremos, en condiciones
  normales la correlación es baja, aportando información genuinamente
  independiente al encoder.
- **Centrado en cero:** Por construcción, z_vol_t tiene media aproximadamente
  cero en cualquier ventana temporal, lo que facilita su normalización
  conjunta con las demás features.

In [ ]:
# ── Cómputo del z-score del volumen ──────────────────────────────────────────
def compute_volume_zscore(df, window=21):
    """
    Calcula el z-score del log-volumen respecto a su media y std rodantes.

    Parámetros
    ----------
    df     : pd.DataFrame con columna 'Volume'
    window : int, tamaño de la ventana rodante (default=21)

    Retorna
    -------
    pd.Series con el z-score del volumen

    Notas
    -----
    - Se aplica log1p antes de la estandarización para corregir asimetría.
    - La ventana rodante es estrictamente hacia atrás (causal).
    - min_periods=window: los primeros (window-1) valores son NaN.
    """
    log_vol    = np.log1p(df['Volume'])
    roll_mean  = log_vol.rolling(window=window, min_periods=window).mean()
    roll_std   = log_vol.rolling(window=window, min_periods=window).std()

    # Evitar división por cero en sesiones sin variación de volumen
    roll_std   = roll_std.replace(0, np.nan)

    return (log_vol - roll_mean) / roll_std

for ticker in TICKERS:
    dfs[ticker]['vol_zscore'] = compute_volume_zscore(dfs[ticker],
                                                       window=ROLL_WIN)

# ── Verificación numérica ─────────────────────────────────────────────────────
print("VERIFICACIÓN: Z-SCORE DEL VOLUMEN")
print("=" * 75)
print(f"{'Activo':<14}  {'Media':>8}  {'Std':>8}  "
      f"{'Min':>8}  {'Max':>8}  {'NaN':>6}  {'|z|>3 (%)':>12}")
print('-' * 70)

for ticker in TICKERS:
    z = dfs[ticker]['vol_zscore'].dropna()
    nans   = dfs[ticker]['vol_zscore'].isna().sum()
    pct_ex = (np.abs(z) > 3).sum() / len(z) * 100
    print(f"{NAMES[ticker]:<14}  {z.mean():>8.4f}  {z.std():>8.4f}  "
          f"{z.min():>8.4f}  {z.max():>8.4f}  {nans:>6}  {pct_ex:>11.2f}%")


In [ ]:
# ── Distribución del log-volumen: crudo vs transformado ──────────────────────
fig, axes = plt.subplots(2, len(TICKERS), figsize=(18, 7))

for j, ticker in enumerate(TICKERS):
    vol_raw = dfs[ticker]['Volume'].dropna()
    vol_log = np.log1p(vol_raw)

    # Volumen bruto
    axes[0, j].hist(vol_raw, bins=60, color=COLORS[ticker],
                    alpha=0.7, edgecolor='none', density=True)
    axes[0, j].set_title(f'{NAMES[ticker]}
Volumen bruto', fontweight='bold')
    axes[0, j].set_xlabel('Volumen (acciones)')
    skew_raw = stats.skew(vol_raw)
    axes[0, j].text(0.65, 0.90, f'Asimetría: {skew_raw:.2f}',
                    transform=axes[0, j].transAxes, fontsize=9,
                    bbox=dict(boxstyle='round', fc='white', alpha=0.8))

    # Log-volumen
    axes[1, j].hist(vol_log, bins=60, color=COLORS[ticker],
                    alpha=0.7, edgecolor='none', density=True)
    axes[1, j].set_title(f'{NAMES[ticker]}
Log-volumen', fontweight='bold')
    axes[1, j].set_xlabel('ln(1 + Volumen)')
    skew_log = stats.skew(vol_log)
    axes[1, j].text(0.05, 0.90, f'Asimetría: {skew_log:.2f}',
                    transform=axes[1, j].transAxes, fontsize=9,
                    bbox=dict(boxstyle='round', fc='white', alpha=0.8))

plt.suptitle('Feature 3: Corrección de Asimetría — Volumen Bruto vs Log-Volumen',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_f3_vol_transformation.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Z-score del volumen en el tiempo — panel interactivo ─────────────────────
fig = make_subplots(
    rows=len(TICKERS), cols=1,
    shared_xaxes=True,
    subplot_titles=[f'{NAMES[t]} — Z-Score Volumen' for t in TICKERS],
    vertical_spacing=0.05
)

for i, ticker in enumerate(TICKERS, start=1):
    d = dfs[ticker].dropna(subset=['vol_zscore'])

    fig.add_trace(
        go.Scatter(x=d.index, y=d['vol_zscore'],
                   mode='lines', name=NAMES[ticker],
                   line=dict(color=COLORS[ticker], width=0.8)),
        row=i, col=1
    )
    # Línea ±3
    fig.add_hline(y= 3, line_dash='dash', line_color='grey',
                  line_width=0.8, row=i, col=1)
    fig.add_hline(y=-3, line_dash='dash', line_color='grey',
                  line_width=0.8, row=i, col=1)
    fig.add_hline(y= 0, line_dash='dot', line_color='black',
                  line_width=0.5, row=i, col=1)

    for ap in ANOMALY_PERIODS:
        fig.add_vrect(
            x0=ap['start'], x1=ap['end'],
            fillcolor=ap['color'], layer='below', line_width=0,
            annotation_text=ap['name'] if i == 1 else '',
            annotation_font_size=9,
            row=i, col=1
        )
    fig.update_yaxes(title_text='z_vol', row=i, col=1)

fig.update_layout(
    title_text='<b>Feature 3: Z-Score del Volumen — ADRs Colombianos</b>',
    height=270 * len(TICKERS), width=1100,
    template='plotly_white'
)
fig.show()


In [ ]:
# ── Correlación entre |retorno| y z-score del volumen ────────────────────────
print("CORRELACIÓN: |RETORNO| vs Z-SCORE DEL VOLUMEN")
print("=" * 60)
print()
print("Régimen normal (2015–2019):")
print(f"  {'Activo':<14}  {'Correlación Pearson':>22}  {'p-valor':>12}")
print('-' * 52)

for ticker in TICKERS:
    d = dfs[ticker].loc['2015-01-01':'2019-12-31'].dropna(
        subset=['log_return', 'vol_zscore'])
    r_abs = d['log_return'].abs()
    z     = d['vol_zscore']
    corr, pval = stats.pearsonr(r_abs, z)
    print(f"  {NAMES[ticker]:<14}  {corr:>22.4f}  {pval:>12.6f}")

print()
print("Régimen crisis COVID-19 (feb–may 2020):")
print(f"  {'Activo':<14}  {'Correlación Pearson':>22}  {'p-valor':>12}")
print('-' * 52)

for ticker in TICKERS:
    d = dfs[ticker].loc['2020-02-15':'2020-05-01'].dropna(
        subset=['log_return', 'vol_zscore'])
    if len(d) < 5:
        print(f"  {NAMES[ticker]:<14}  {'insuf. datos':>22}")
        continue
    r_abs = d['log_return'].abs()
    z     = d['vol_zscore']
    corr, pval = stats.pearsonr(r_abs, z)
    print(f"  {NAMES[ticker]:<14}  {corr:>22.4f}  {pval:>12.6f}")

print()
print("Nota: la correlacion aumenta durante crisis, confirmando que el")
print("volumen anonmalo y el retorno extremo ocurren conjuntamente en crisis.")


### **4.4 Decisión de preprocesamiento**

| Aspecto | Decisión |
|---|---|
| Transformación previa | `log_vol = log1p(Volume)` |
| Fórmula z-score | `(log_vol - rolling_mean(21)) / rolling_std(21)` |
| Ventana | 21 sesiones (misma que volatilidad, coherencia temporal) |
| Primeros 20 valores | NaN — descartar junto con los de vol_21d |
| Normalización adicional | StandardScaler sobre valores ya z-scored — opcional |
| Inclusión en el modelo | Tercera columna del vector de features |

### **Riesgos identificados**

- **Sesiones con volumen cero:** Si existieran sesiones sin negociación
  que no fueron descartadas en el preprocesamiento, generarían z_vol_t = −∞.
  **Control:** Verificar que `Volume > 0` en todos los registros tras la descarga.
- **Inestabilidad del z-score con σ ≈ 0:** Si todos los volúmenes en la
  ventana rodante fueran idénticos, σ_t = 0 y el z-score sería indefinido.
  **Control:** Reemplazar σ_t = 0 con NaN antes de la división.
- **Comparabilidad entre activos:** Los z-scores de distintos activos no
  son directamente comparables en magnitud (TGLS tiene volúmenes menores
  y más variables). Para el modelo multivariado, el StandardScaler posterior
  equaliza la escala entre activos.

---
## Validación del Vector de Features

### **Motivación**
Antes de proceder con la normalización y el windowing, es necesario validar
que el vector de features cumple los cuatro criterios de diseño enunciados
en la introducción: estacionariedad, causalidad, independencia informativa
y relevancia para la detección de anomalías.

In [ ]:
# ── Construcción del DataFrame de features por activo ────────────────────────
feature_dfs = {}

for ticker in TICKERS:
    d = dfs[ticker][FEATURE_COLS].copy()
    # Eliminar filas con cualquier NaN (período de calentamiento)
    d = d.dropna()
    feature_dfs[ticker] = d

print("RESUMEN DEL VECTOR DE FEATURES DESPUÉS DE DESCARTAR NaN")
print("=" * 65)
print(f"{'Activo':<14}  {'Inicio':>12}  {'Fin':>12}  {'Sesiones':>10}")
print('-' * 55)
for ticker in TICKERS:
    d = feature_dfs[ticker]
    print(f"{NAMES[ticker]:<14}  "
          f"{str(d.index.min().date()):>12}  "
          f"{str(d.index.max().date()):>12}  "
          f"{len(d):>10}")


In [ ]:
# ── Matriz de correlación entre features — período de entrenamiento ───────────
fig, axes = plt.subplots(1, len(TICKERS), figsize=(18, 4))

for j, ticker in enumerate(TICKERS):
    d_train = feature_dfs[ticker].loc[:TRAIN_END]
    corr    = d_train.corr()

    labels = ['r_t', 'σ_t', 'z_vol']
    sns.heatmap(
        corr, ax=axes[j], cmap='coolwarm', center=0,
        vmin=-1, vmax=1, annot=True, fmt='.3f',
        linewidths=0.5, cbar=(j == len(TICKERS) - 1),
        xticklabels=labels, yticklabels=labels
    )
    axes[j].set_title(f'{NAMES[ticker]}
(2015–2019)', fontweight='bold')
    axes[j].tick_params(axis='x', labelsize=9)
    axes[j].tick_params(axis='y', labelsize=9, rotation=0)

plt.suptitle(
    'Matriz de Correlación entre Features — Período de Entrenamiento',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('fig_feature_correlation.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Test ADF sobre cada feature ───────────────────────────────────────────────
from statsmodels.tsa.stattools import adfuller

print("ESTACIONARIEDAD DE CADA FEATURE — TEST ADF (α = 5%)")
print("=" * 70)
print(f"{'Feature':<12}  {'Activo':<14}  {'ADF stat':>10}  "
      f"{'p-valor':>10}  {'Estac.':>8}")
print('-' * 58)

for feat in FEATURE_COLS:
    for ticker in TICKERS:
        series = feature_dfs[ticker][feat].dropna()
        res    = adfuller(series, autolag='AIC')
        estac  = 'SI' if res[1] < 0.05 else 'NO'
        print(f"{feat:<12}  {NAMES[ticker]:<14}  "
              f"{res[0]:>10.4f}  {res[1]:>10.6f}  {estac:>8}")
    print()


In [ ]:
# ── Visualización conjunta: 3 features en el tiempo (CIB como ejemplo) ────────
ticker_demo = 'CIB'
d_demo      = feature_dfs[ticker_demo]

fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)
feat_labels = {
    'log_return': ('Log-Retorno (r_t)',      COLORS[ticker_demo]),
    'vol_21d':    ('Volatilidad 21d (σ_t)',  '#9467bd'),
    'vol_zscore': ('Z-Score Volumen (z_vol)', '#8c564b'),
}

for ax, (feat, (label, color)) in zip(axes, feat_labels.items()):
    ax.plot(d_demo.index, d_demo[feat].values,
            color=color, linewidth=0.7, alpha=0.85)
    ax.axhline(0, color='black', linewidth=0.4, alpha=0.4)
    ax.set_ylabel(label, fontsize=9)

    for ap in ANOMALY_PERIODS:
        ax.axvspan(pd.Timestamp(ap['start']),
                   pd.Timestamp(ap['end']),
                   alpha=0.10, color='red',
                   label=ap['name'] if ax == axes[0] else '')

axes[0].legend(fontsize=8, loc='upper right')
axes[0].set_title(f'{NAMES[ticker_demo]} — Vector de Features Completo (2015–2024)',
                  fontweight='bold')

plt.suptitle('Feature 1 (r_t) + Feature 2 (σ_t) + Feature 3 (z_vol)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('fig_feature_vector_completo.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Comportamiento medio de cada feature en cada régimen ─────────────────────
print("COMPORTAMIENTO MEDIO DE FEATURES POR RÉGIMEN — EC (Ecopetrol)")
print("=" * 68)

ticker_r = 'EC'
regimes_display = {
    'Normal 2015-2019': ('2015-01-01', '2019-12-31'),
    'Crisis petroleo':   ('2015-07-01', '2016-02-01'),
    'COVID-19':          ('2020-02-15', '2020-05-01'),
    'Fed Hikes':         ('2022-01-01', '2022-12-31'),
}

print(f"{'Régimen':<22}  {'|r_t| medio':>14}  "
      f"{'σ_t medio':>12}  {'z_vol medio':>14}")
print('-' * 66)

for regime, (s, e) in regimes_display.items():
    d = feature_dfs[ticker_r].loc[s:e]
    print(f"{regime:<22}  "
          f"{d['log_return'].abs().mean():>14.5f}  "
          f"{d['vol_21d'].mean():>12.4f}  "
          f"{d['vol_zscore'].mean():>14.4f}")

print()
print("Conclusion: durante crisis, los tres indicadores se elevan simultaneamente.")
print("Esta co-elevacion multivariate es la señal que el Autoencoder debe capturar.")


---
## Normalización sin Data Leakage

### **Motivación**
El StandardScaler estandariza cada feature a media 0 y desviación estándar 1.
Su ajuste debe realizarse **exclusivamente sobre el conjunto de entrenamiento**
para evitar que las estadísticas de la transformación incorporen información
de los periodos de validación o test, incluyendo la mayor varianza de los
periodos de crisis.

In [ ]:
# ── Pipeline de normalización — sin data leakage ─────────────────────────────

def build_splits(feature_df, train_end, val_start, val_end,
                 test_start, test_end):
    """
    Particiona el DataFrame de features en train, val y test
    con corte cronológico estricto.

    Retorna
    -------
    dict con keys 'train', 'val', 'test' → pd.DataFrames
    """
    return {
        'train': feature_df.loc[: train_end],
        'val':   feature_df.loc[val_start  : val_end],
        'test':  feature_df.loc[test_start : test_end],
    }


def fit_and_scale(splits):
    """
    Ajusta StandardScaler sobre el split de entrenamiento
    y transforma los tres splits.

    Reglas anti-leakage:
    - scaler.fit() → SOLO sobre train
    - scaler.transform() → sobre train, val y test por separado

    Retorna
    -------
    scaled_splits : dict con arrays escalados (numpy)
    scaler        : objeto ajustado (para persistencia)
    """
    scaler = StandardScaler()

    # Ajuste exclusivo en entrenamiento
    scaler.fit(splits['train'].values)

    scaled = {}
    for split_name, df in splits.items():
        scaled[split_name] = scaler.transform(df.values)

    return scaled, scaler


# Aplicar a cada activo
scalers       = {}
scaled_splits = {}
raw_splits    = {}

for ticker in TICKERS:
    splits = build_splits(
        feature_dfs[ticker],
        TRAIN_END, VAL_START, VAL_END, TEST_START, TEST_END
    )
    raw_splits[ticker] = splits
    sc_splits, scaler  = fit_and_scale(splits)
    scaled_splits[ticker] = sc_splits
    scalers[ticker]       = scaler

print("ESTADÍSTICAS DEL SCALER (ajustado en entrenamiento)")
print("=" * 65)
print(f"{'Activo':<14}  {'Feature':<12}  {'μ (train)':>12}  {'σ (train)':>12}")
print('-' * 55)

for ticker in TICKERS:
    sc = scalers[ticker]
    for i, feat in enumerate(FEATURE_COLS):
        print(f"{NAMES[ticker]:<14}  {feat:<12}  "
              f"{sc.mean_[i]:>12.6f}  {sc.scale_[i]:>12.6f}")
    print()


In [ ]:
# ── Verificación: distribución antes y después de escalar ────────────────────
ticker_v = 'EC'

fig, axes = plt.subplots(2, len(FEATURE_COLS), figsize=(16, 7))

for j, feat in enumerate(FEATURE_COLS):
    raw_train = raw_splits[ticker_v]['train'][feat].values
    sc_train  = scaled_splits[ticker_v]['train'][:, j]

    # Antes
    axes[0, j].hist(raw_train, bins=60, color=COLORS[ticker_v],
                    alpha=0.7, density=True, edgecolor='none')
    axes[0, j].set_title(f'{feat}
(sin escalar)', fontweight='bold')
    axes[0, j].set_xlabel('Valor original')
    mu_r, sig_r = raw_train.mean(), raw_train.std()
    axes[0, j].text(0.05, 0.92, f'μ={mu_r:.4f}\nσ={sig_r:.4f}',
                    transform=axes[0, j].transAxes, fontsize=9,
                    va='top',
                    bbox=dict(boxstyle='round', fc='white', alpha=0.8))

    # Después
    axes[1, j].hist(sc_train, bins=60, color='#7f7f7f',
                    alpha=0.7, density=True, edgecolor='none')
    axes[1, j].set_title(f'{feat}
(StandardScaler)', fontweight='bold')
    axes[1, j].set_xlabel('Valor escalado')
    mu_s, sig_s = sc_train.mean(), sc_train.std()
    axes[1, j].text(0.05, 0.92, f'μ={mu_s:.4f}\nσ={sig_s:.4f}',
                    transform=axes[1, j].transAxes, fontsize=9,
                    va='top',
                    bbox=dict(boxstyle='round', fc='white', alpha=0.8))

plt.suptitle(
    f'{NAMES[ticker_v]} — Distribución de Features: Antes y Después del Escalado',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('fig_scaling_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Verificar no-leakage: comparar media val/test escalados vs 0 ──────────────
print("AUDITORÍA ANTI-LEAKAGE")
print("=" * 65)
print("Si el scaler se hubiera ajustado en el conjunto completo,")
print("la media escalada de train sería ≈ 0 pero también la de val/test.")
print("Con ajuste correcto (solo train), val y test pueden tener")
print("media != 0 (especialmente en periodos de crisis con mayor volatilidad).")
print()

for ticker in ['EC', 'CIB']:
    print(f"{NAMES[ticker]}:")
    for split_name in ['train', 'val', 'test']:
        arr  = scaled_splits[ticker][split_name]
        means = arr.mean(axis=0)
        line  = "  " + f"{split_name:<8}" + "  "
        for i, feat in enumerate(FEATURE_COLS):
            line += f"{feat}={means[i]:+.4f}  "
        print(line)
    print()

print("Interpretacion esperada:")
print("  - train: medias ≈ 0.0000 (por construccion del StandardScaler)")
print("  - val:   medias desviadas de 0 (COVID — mayor volatilidad)")
print("  - test:  medias moderadamente desviadas de 0")
print("  Si val y test tuvieran media ≈ 0, habria leakage.")


### **Decisión de normalización — resumen**

| Aspecto | Decisión |
|---|---|
| Método | `sklearn.preprocessing.StandardScaler` |
| Ajuste | Exclusivamente sobre `train` (2015–2019) |
| Aplicación | `.transform()` sobre train, val y test por separado |
| Persistencia | El objeto `scaler` se guarda junto con los pesos del modelo |
| Verificación | La media escalada de val/test debe ser distinta de 0 en features de volatilidad |

### **Riesgos identificados**

- **Leakage de escala (el más común):** Usar `scaler.fit(X_completo)` antes
  de particionar incorpora la varianza de las crisis al denominador,
  reduciendo artificialmente la escala de las features de crisis en el test set
  y comprimiendo la señal de anomalía. **Control:** La auditoría anterior
  verifica que las medias escaladas de val/test son distintas de cero.
- **Leakage por reinicialización:** Si el modelo se re-entrena en producción
  con datos más recientes, el scaler debe re-ajustarse sobre el nuevo conjunto
  de entrenamiento. Usar el scaler original con datos de una ventana temporal
  diferente introduce sesgo de escala.

---
## Generación de Ventanas Temporales

### **Motivación**
El Autoencoder LSTM recibe como entrada tensores de forma `(batch, T, F)`.
La función de windowing transforma el array escalado 2D `(N, F)` en un
tensor 3D `(N−T+1, T, F)` con ventanas deslizantes de longitud T.

In [ ]:
# ── Función de windowing ──────────────────────────────────────────────────────
def create_windows(array, seq_len):
    """
    Genera ventanas deslizantes de longitud seq_len sobre un array 2D.

    Parámetros
    ----------
    array   : np.ndarray de forma (N, F)
    seq_len : int, longitud de cada ventana

    Retorna
    -------
    np.ndarray de forma (N - seq_len + 1, seq_len, F)

    Notas
    -----
    - Stride = 1 (ventana deslizante de un día).
    - La ventana i corresponde a las observaciones [i, i + seq_len).
    - No existe solapamiento entre ventanas de distintos splits.
    """
    n_windows = array.shape[0] - seq_len + 1
    if n_windows <= 0:
        raise ValueError(
            f"El array tiene {array.shape[0]} filas pero se requieren "
            f"al menos {seq_len} para formar una ventana."
        )

    # Construcción eficiente mediante stride_tricks
    windows = np.lib.stride_tricks.sliding_window_view(
        array, window_shape=(seq_len, array.shape[1])
    ).squeeze(axis=1)          # (N-T+1, T, F)

    return windows.copy()       # copia para evitar referencias a memoria compartida


# ── Aplicar a todos los activos y splits ─────────────────────────────────────
windows = {}   # windows[ticker][split] = np.ndarray (n_win, T, F)

for ticker in TICKERS:
    windows[ticker] = {}
    for split_name in ['train', 'val', 'test']:
        arr = scaled_splits[ticker][split_name]
        windows[ticker][split_name] = create_windows(arr, SEQ_LEN)

# ── Resumen de shapes ─────────────────────────────────────────────────────────
print(f"SHAPES DE LOS TENSORES DE WINDOWS (T={SEQ_LEN}, F={len(FEATURE_COLS)})")
print("=" * 65)
print(f"{'Activo':<14}  {'Train':>18}  {'Val':>18}  {'Test':>18}")
print('-' * 70)

for ticker in TICKERS:
    shapes = {s: windows[ticker][s].shape for s in ['train', 'val', 'test']}
    print(f"{NAMES[ticker]:<14}  "
          f"{str(shapes['train']):>18}  "
          f"{str(shapes['val']):>18}  "
          f"{str(shapes['test']):>18}")


In [ ]:
# ── Verificación de la estructura de una ventana ─────────────────────────────
ticker_v2 = 'CIB'
w_demo    = windows[ticker_v2]['train']

print(f"INSPECCION DE VENTANA — {NAMES[ticker_v2]} — Entrenamiento")
print("=" * 55)
print(f"Shape del tensor train: {w_demo.shape}")
print(f"  Dimensión 0 (ventanas)   : {w_demo.shape[0]}")
print(f"  Dimensión 1 (timesteps)  : {w_demo.shape[1]}  (= T = {SEQ_LEN})")
print(f"  Dimensión 2 (features)   : {w_demo.shape[2]}  "
      f"(= {FEATURE_COLS})")

print()
print("Ventana 0 (primeras 5 filas):")
print(f"  {'Timestep':>10}  {'log_return':>12}  {'vol_21d':>12}  {'vol_zscore':>12}")
print('-' * 52)
for t in range(5):
    print(f"  {t:>10}  "
          f"{w_demo[0, t, 0]:>12.4f}  "
          f"{w_demo[0, t, 1]:>12.4f}  "
          f"{w_demo[0, t, 2]:>12.4f}")
print("  ...")

print()
print("Ultima ventana del train (primeras 3 filas):")
last_idx = w_demo.shape[0] - 1
for t in range(3):
    print(f"  t={t:<4}  "
          f"log_ret={w_demo[last_idx, t, 0]:>8.4f}  "
          f"vol={w_demo[last_idx, t, 1]:>8.4f}  "
          f"zvol={w_demo[last_idx, t, 2]:>8.4f}")

print()
print("Verificación de no-solapamiento entre train y val:")
n_train = scaled_splits[ticker_v2]['train'].shape[0]
print(f"  Última ventana train termina en índice: {n_train - 1}")
print(f"  Primera ventana val  empieza en índice: {n_train} (del split val)")
print("  No existe solapamiento. OK.")


---
## Features y Detección de Cambios de Régimen

### **Definición de cambio de régimen**

Un cambio de régimen en el contexto de mercados financieros se define como
una transición estructural en las propiedades estadísticas del proceso
generador de datos que subyace a los precios de mercado. Formalmente:

```
Régimen R_k = {proceso estocástico con parámetros θ_k}

Cambio de régimen: θ_{k+1} ≠ θ_k  en el instante τ_k
```

Los parámetros que cambian típicamente en una crisis son:
- **μ (media de retornos):** Baja significativamente o se vuelve negativa.
- **σ (volatilidad):** Aumenta abruptamente (efecto leverage).
- **ρ (correlación entre activos):** Aumenta (contagio de correlación).
- **γ (asimetría de la distribución):** Se hace más negativa.
- **κ (curtosis):** Aumenta (eventos extremos más frecuentes).

### **Cómo el vector de features captura los cambios de régimen**

El vector [r_t, σ_t, z_vol_t] está diseñado para ser sensible a cada una
de estas dimensiones del cambio:

In [ ]:
# ── Análisis de cambio de régimen: distribuciones por periodo ─────────────────
ticker_r = 'EC'
d_full   = feature_dfs[ticker_r]

regimes_comp = {
    'Normal\n(2017-2019)': ('2017-01-01', '2019-12-31', '#1f77b4'),
    'Crisis petroleo\n(2015-2016)': ('2015-07-01', '2016-02-01', '#9467bd'),
    'COVID-19\n(Feb-May 2020)': ('2020-02-15', '2020-05-01', '#d62728'),
    'Fed Hikes\n(2022)': ('2022-01-01', '2022-12-31', '#ff7f0e'),
}

fig, axes = plt.subplots(len(FEATURE_COLS), len(regimes_comp),
                         figsize=(18, 9), sharex='row')

for row, feat in enumerate(FEATURE_COLS):
    for col, (regime_name, (s, e, color)) in             enumerate(regimes_comp.items()):
        data_r = d_full.loc[s:e, feat].dropna()

        axes[row, col].hist(data_r, bins=40, density=True,
                            color=color, alpha=0.75, edgecolor='none')
        axes[row, col].axvline(data_r.mean(), color='black',
                               linewidth=1.2, linestyle='--',
                               label=f'μ={data_r.mean():.3f}')
        if row == 0:
            axes[row, col].set_title(regime_name, fontweight='bold',
                                     fontsize=10)
        if col == 0:
            axes[row, col].set_ylabel(feat, fontweight='bold')

        axes[row, col].legend(fontsize=7)

plt.suptitle(
    f'{NAMES[ticker_r]} — Distribución de Features por Régimen de Mercado',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('fig_regime_distributions.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── Score de anomalía heurístico — suma ponderada de desviaciones ─────────────
# NOTA: este no es el score del modelo (que es el MSE de reconstruccion),
# sino un proxy analítico para ilustrar cómo el vector de features captura
# las anomalías antes de entrenar el modelo.

def heuristic_anomaly_score(df_feat, scaler_obj):
    """
    Calcula un score de anomalía heurístico como la distancia L2 normalizada
    del vector de features respecto a la media del entrenamiento.

    Este indicador NO reemplaza al MSE del Autoencoder; sirve para
    visualizar la separabilidad de los regímenes en el espacio de features.
    """
    # Escalar con el scaler ya ajustado
    scaled = scaler_obj.transform(df_feat.values)
    # Distancia L2 al origen (que tras escalar corresponde a la media de train)
    score  = np.sqrt((scaled ** 2).sum(axis=1))
    return pd.Series(score, index=df_feat.index)

fig = make_subplots(
    rows=len(TICKERS), cols=1,
    shared_xaxes=True,
    subplot_titles=[
        f'{NAMES[t]} — Score Heurístico de Anomalía (L2 sobre features escaladas)'
        for t in TICKERS
    ],
    vertical_spacing=0.05
)

for i, ticker in enumerate(TICKERS, start=1):
    score = heuristic_anomaly_score(
        feature_dfs[ticker], scalers[ticker]
    )

    # Percentil 95 del período de entrenamiento como umbral visual
    tau = np.percentile(
        score.loc[:TRAIN_END].values, 95
    )

    fig.add_trace(
        go.Scatter(x=score.index, y=score.values,
                   mode='lines', name=NAMES[ticker],
                   line=dict(color=COLORS[ticker], width=0.9)),
        row=i, col=1
    )
    fig.add_hline(y=tau, line_dash='dash', line_color='red',
                  line_width=1.2,
                  annotation_text=f'τ (p95 train) = {tau:.2f}',
                  annotation_font_size=9,
                  row=i, col=1)

    for ap in ANOMALY_PERIODS:
        fig.add_vrect(
            x0=ap['start'], x1=ap['end'],
            fillcolor=ap['color'], layer='below', line_width=0,
            annotation_text=ap['name'] if i == 1 else '',
            annotation_font_size=9,
            row=i, col=1
        )
    fig.update_yaxes(title_text='||z||₂', row=i, col=1)

fig.update_layout(
    title_text='<b>Score Heurístico de Anomalía — Distancia L2 en Espacio de Features</b>',
    height=280 * len(TICKERS), width=1100,
    template='plotly_white'
)
fig.show()


In [ ]:
# ── Análisis cuantitativo: score por encima del umbral en cada régimen ────────
print("TASA DE DETECCION HEURÍSTICA (score > p95 train) POR RÉGIMEN")
print("=" * 68)

periodos_eval = {
    'Normal 2015-2019': ('2015-01-01', '2019-12-31'),
    'Crisis petróleo':   ('2015-07-01', '2016-02-01'),
    'COVID-19':          ('2020-02-15', '2020-05-01'),
    'Fed Hikes 2022':    ('2022-01-01', '2022-12-31'),
}

print(f"{'Periodo':<22}", end='')
for t in TICKERS:
    print(f"  {NAMES[t]:>14}", end='')
print()
print('-' * (22 + 16 * len(TICKERS)))

for periodo, (s, e) in periodos_eval.items():
    print(f"{periodo:<22}", end='')
    for ticker in TICKERS:
        score = heuristic_anomaly_score(
            feature_dfs[ticker], scalers[ticker]
        )
        tau   = np.percentile(score.loc[:TRAIN_END].values, 95)
        s_per = score.loc[s:e]
        if len(s_per) == 0:
            print(f"  {'N/A':>14}", end='')
        else:
            pct = (s_per > tau).sum() / len(s_per) * 100
            print(f"  {pct:>13.1f}%", end='')
    print()

print()
print("Interpretacion:")
print("  Normal: ~5% (por definicion del percentil 95)")
print("  Crisis: >> 5% confirma separabilidad de regimenes en el espacio de features")


---

### **Interpretación técnica — Cómo los features capturan los cambios de régimen**

#### **Feature 1 — r_t (Log-Retorno)**

El retorno logarítmico es la señal más inmediata de un cambio de régimen.
Durante crisis, la distribución de r_t se desplaza hacia la izquierda (media
negativa) y sus colas se vuelven más pesadas. El Autoencoder aprende que las
secuencias normales tienen retornos cercanos a cero con varianza acotada.
Cualquier ventana con retornos persistentemente negativos o de magnitud
excepcional genera un error de reconstrucción elevado porque el encoder no
puede comprimir ese patrón en el espacio latente que aprendió del comportamiento
normal.

#### **Feature 2 — σ_t (Volatilidad Realizada 21d)**

La volatilidad realizada captura el **estado del régimen** y no sólo el evento
puntual. Su clave es la **persistencia temporal:** durante una crisis, σ_t se
eleva durante semanas consecutivas, no sólo en el día del evento. Esto significa
que incluso si una ventana de 30 días no contiene ningún retorno extremo
individual pero sí una σ_t persistentemente elevada (por ejemplo, la segunda
mitad de un periodo de crisis cuando los retornos diarios se moderan pero la
volatilidad sigue alta), el Autoencoder la reconstruirá mal porque esa
combinación de features no estaba presente durante el entrenamiento normal.

#### **Feature 3 — z_vol_t (Z-Score del Volumen)**

El volumen anómalo es un indicador líder de cambios de régimen. En muchos
eventos de mercado, el volumen se eleva antes de que el movimiento de precios
sea plenamente visible en los retornos. La co-ocurrencia de z_vol elevado con
retornos moderados puede señalar el inicio de un cambio de régimen antes de
que σ_t lo refleje (dado el rezago de 21 días de la ventana de volatilidad).
Adicionalmente, z_vol distingue entre movimientos de precio acompañados de
alta participación (eventos genuinos) y movimientos de precio en condiciones
de baja liquidez (mayor probabilidad de reversión, menor relevancia sistémica).

#### **La señal multivariada**

La mayor potencia detectora no proviene de ningún feature individual sino de
la **combinación multivariada** en el espacio latente del Autoencoder. El
encoder aprende la distribución conjunta normal de (r_t, σ_t, z_vol_t) en
ventanas de 30 días. Una anomalía en el espacio multivariado puede ser invisible
en cualquier feature individual pero visible en su correlación: por ejemplo,
un retorno moderado combinado con volumen extremadamente alto y volatilidad
baja es una configuración que no aparece en el training set normal y generará
un error de reconstrucción elevado.

---

### **Resumen del vector de features**

| Feature | Fórmula | Señal capturada | Horizonte |
|---|---|---|---|
| `log_return` | ln(P_t / P_{t-1}) | Evento puntual de precio | Instantáneo |
| `vol_21d` | rolling_std(r, 21) × √252 | Estado del régimen de volatilidad | 21 sesiones |
| `vol_zscore` | (log_vol − μ_{21d}) / σ_{21d} | Actividad de mercado anómala | 21 sesiones |

**Forma del tensor de entrada al modelo:**

```python
X.shape == (n_windows, seq_len=30, n_features=3)
```

**Próximo paso:** `Notebook_03_Modelo_Autoencoder.ipynb` — construcción,
entrenamiento y evaluación del Denoising Autoencoder LSTM/GRU sobre los tensores
generados en este notebook.